# Mesurer la performance environnementale des pays : analyse critique et alternatives à l'EPI

*Projet Python pour la Data Science — Giorgi KUPRADZE, Martin LOUP, Niels SOUFFIR — 2A ENSAE Paris — 2026.*

## Sommaire

- **Introduction**

- **I. Présentation et exploration des données**
    - A. Récupération des données via l'API
    - B. Présentation des bases de données
    - C. Cartographie de l'EPI officiel

- **II. Analyse critique de l'EPI officiel**
    - A. Lien entre richesse et performance environnementale (économétrie)
    - B. Diversité des profils environnementaux (K-means)
    - C. Structure statistique et redondance des catégories (ACP)

- **III. Construction de méthodes alternatives de comparaison**
    - A. Pondérations fondées sur les données (ACP)
    - B. Comparaison sans pondération (fronts de Pareto)
    - C. Redéfinition des pondérations centrées sur les besoins humains

- **IV. Analyse spatiale et temporelle des performances environnementales**
    - A. Construction et nettoyage des deux bases historiques (1995-2022) : EPI à pondérations officielles et nouvel EPI
    - B. Analyse et comparaison des trajectoires environnementales selon les deux indices

- **V. Conclusion et limites méthodologiques**
    - A. Synthèse des résultats
    - B. Limites des différentes approches
    - C. Perspectives d'amélioration

## Introduction

Face à la multiplication des défis environnementaux, l'évaluation des performances des pays constitue un enjeu majeur. L'Environmental Performance Index (EPI), développé par les universités de Yale et de Columbia, propose une mesure synthétique de ces performances à travers trois grands objectifs : la santé environnementale, la vitalité des écosystèmes et la lutte contre le changement climatique.

La construction d'un indice unique pour comparer 180 pays soulève néanmoins plusieurs interrogations. Les pays présentent des niveaux de développement et des caractéristiques géographiques très différents. Tous les pays n’ont pas les mêmes priorités environnementales : certains doivent répondre à des besoins immédiats, comme l’accès à l’eau potable, tandis que d’autres peuvent se concentrer sur des enjeux de long terme. Accorder la même importance aux différents enjeux environnementaux pour l'ensemble des pays revient donc à faire des choix qui ne sont pas neutres. Cette question est d’autant plus importante que l’EPI, au-delà d’un simple classement international, peut influencer les priorités des gouvernements et les politiques environnementales mises en place.

Ces constats nous amènent à poser la problématique suivante :


**Comment mesurer et comparer les performances environnementales des pays face à la diversité des enjeux environnementaux, notamment du point de vue du bien-être humain ?**


Pour répondre à cette question, nous commencerons par présenter et explorer les données issues de l'édition 2022 de l'etude de l'EPI. Nous procéderons ensuite à une analyse critique de l'indice officiel, en étudiant notamment son lien avec la richesse des pays, la diversité des profils environnementaux et la structure statistique de ses différentes catégories.

Nous explorerons plusieurs méthodes alternatives de comparaison, avant de proposer un nouvel indice privilégiant les besoins humains à court terme. Nous faisons le choix d'une approche volontairement égoïste, considérant que certains problèmes environnementaux ayant un impact direct sur la santé des populations doivent être traités en priorité.

Enfin, nous adopterons une approche dynamique en étudiant les trajectoires environnementales des pays et des régions entre 1995 et 2022. La comparaison de ces trajectoires selon les deux indices nous permettra d'apprécier l'influence des choix de pondération sur l'évaluation des performances environnementales.

In [ ]:
%pip install -r requirements.txt

## **I. Présentation et exploration des données**
## I.A. Récupération des données via l'API de la NASA

In [ ]:
"""First method of data retrieval. Unfortunately, the website doesn't give the necessary permission"""
import os
import requests
import earthaccess
from zipfile import ZipFile

Entrez votre identifiant/mot de passe de EarthAccess après l'exécution de l aprochaine cellule:

In [ ]:
# 1. Login
earthaccess.login()

Une fois le prochain script exécuté, la commande 
```
tree raw_data -d
```
dans bash doit avoir la réponse suivante:
```
raw_data/
├── 2022-epi-indicators-time-series
├── 2022-epi-indicators-time-series-na
├── 2022-epi-raw-data-time-series
└── 2022-epi-raw-data-time-series-na

5 directories
```

In [ ]:
# 2. Search
results = earthaccess.search_data(
    short_name='CIESIN_SEDAC_EPI_2022',
)

# 3. Access
if results != []:
    cwd = os.getcwd()
    files = earthaccess.download(results, f"{cwd}/raw_data/")
    
    # zip folders:
    # 2022-epi-indicators-time-series-csv.zip
    # 2022-epi-indicators-time-series-na-csv.zip
    # 2022-epi-raw-data-time-series-csv.zip
    # 2022-epi-raw-data-time-series-na-csv.zip
    # 2022-epi-xlsx.zip
    
    # WARNING: in raw_data/ we also have PDF files:
    # 2022-epi-report.pdf
    # 2022-epi-technical-appendix.pdf
    # 2022-epi-policymakers-summary.pdf
    
    # Unzip the zipped folders
    zfiles = ["2022-epi-indicators-time-series-csv.zip",
              "2022-epi-indicators-time-series-na-csv.zip",
              "2022-epi-raw-data-time-series-csv.zip",
              "2022-epi-raw-data-time-series-na-csv.zip",
              "2022-epi-xlsx.zip",
            ]
    for zf in zfiles:
        full_path = f"{cwd}/raw_data/{zf}"
        relative_path = f"raw_data/{zf}"
        with ZipFile(full_path, 'r') as zObject:
            zObject.extractall(path=f"{cwd}/raw_data/")
        zObject.close()
else:
    print("\nresults is empty\n")

## I.B. Présentation des bases de données
Commençons par la liste des indices qui rentrent dans le calcul de l'EPI:

![](extra_images/fig1-3.png)

À chaque indicateur que vous voyez ici, est associé un poids comme suit:

![](extra_images/fig_ES-2.png)

Ce qu'il faut retenir sur l'ensemble des données CSV qui ont été téléchargées dans le dossier raw_data/, c'est qu'ils contiennent l'évolution temporelle pour chaque pays de chaque variable. On ne recalcule pas ces dernières, mais on en synthétise un nouvel indicateur qui va reflèter d'une réalité plus concrète que l'indicateur proposé par la NASA.

Notre indicateur se concentre sur les efforts que chaque pays fait sur le plan environnemental pour rendre son territoire plus habitable pour les humains.

Voici donc notre proposition de modification des poids qui se trouvent dans le fichier excel /home/deck/Documents/pydata_ENSAE/raw_data/2022-epi.xlsx

![](extra_images/new_inxs.png)

### Méthode de calcul:

On a supprimé toutes les sous-catégories (sc) de variables qui n'avaent pas de lien avec notre but. Il nous en restait ainsi 8, auxquelles on a ensuite assigné un entier entre 1 et 8 représentant sa priorité dans la liste des indices.

Les poids des trois catégories (c) ont été ramenés à 1/6, et on a distribué $\epsilon = 0.5$ sur les sous-catégories comme suit:
$$
poids(sc) = w \cdot W + \epsilon \cdot \delta_{0} \cdot 1.1^{8 - priorité}
$$
où
+ w: poids de la sc au sein de la c.
+ W poids de la c, soit 1/6
+ $\epsilon$: poids à départager
+ $\delta_{0}$: solution de l'équation
$$
\delta_{0} \cdot \sum_{i = 0}^{7} 1.1^{i} = 1
$$
On a donc distribué $\epsilon$ de telle façon que chaque sc pèse 10% de plus que la précédente dans la liste des priorités.



In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np

# Hierarchical structure from the spreadsheet:
# Policy Objective -> Groups -> (Variable name, Final weight)
hierarchy = [
    {
        "objective": "Ecosystem Vitality",
        "obj_color": "#81D41A",
        "groups": [
            {
                "group": "Ecosystem Services",
                "grp_color": "#BBE33D",
                "var_color": "#E6F4B8",
                "vars": [
                    ("Tree cover loss", 0.102719359620898),
                    ("Grassland loss", 0.0171198932701496),
                    ("Wetland loss", 0.0171198932701496),
                ],
            },
            {
                "group": "Agriculture",
                "grp_color": "#A3D936",
                "var_color": "#ECF7C8",
                "vars": [
                    ("Sustainable Pesticide Use", 0.0542264301946445),
                    ("Sustainable Nitrogen Management Index", 0.0542264301946445),
                ],
            },
            {
                "group": "Water Resources",
                "grp_color": "#8BC92E",
                "var_color": "#F2FAD8",
                "vars": [
                    ("Wastewater treatment", 0.0862658934807329),
                ],
            },
        ],
    },
    {
        "objective": "Environmental Health",
        "obj_color": "#2A6099",
        "groups": [
            {
                "group": "Air Quality",
                "grp_color": "#5983B0",
                "var_color": "#B8CCE0",
                "vars": [
                    ("PM2.5 exposure", 0.0836057365640669),
                    ("Household solid fuels", 0.0675283905651803),
                    ("Ozone exposure", 0.00803867299944329),
                    ("NOx exposure", 0.00803867299944329),
                    ("SO2 exposure", 0.00321900656963406),
                    ("CO exposure", 0.00321900656963406),
                    ("VOC exposure", 0.00321900656963406),
                ],
            },
            {
                "group": "Sanitation & Drinking Water",
                "grp_color": "#4874A3",
                "var_color": "#C5D6E8",
                "vars": [
                    ("Unsafe sanitation", 0.0448325596155492),
                    ("Unsafe drinking water", 0.0672488394233238),
                ],
            },
            {
                "group": "Heavy Metals",
                "grp_color": "#3A6696",
                "var_color": "#D2E0EE",
                "vars": [
                    ("Lead exposure", 0.0941228722760937),
                ],
            },
            {
                "group": "Waste Management",
                "grp_color": "#6B92BD",
                "var_color": "#DEE6EF",
                "vars": [
                    ("Controlled solid waste", 0.0374303301813525),
                    ("Ocean Plastics", 0.0374303301813525),
                ],
            },
        ],
    },
    {
        "objective": "Climate Change Policy Objective",
        "obj_color": "#FFBF00",
        "groups": [
            {
                "group": "Climate Change",
                "grp_color": "#FFD428",
                "var_color": "#FFF5CE",
                "vars": [
                    ("Adjusted emissions growth rate for carbon dioxide", 0.0764068552646557),
                    ("Adjusted emissions growth rate for methane", 0.0182701525764317),
                    ("Adjusted emissions growth rate for F-gases", 0.00775071880372805),
                    ("Adjusted emissions growth rate for nitrous oxide", 0.00387535940186402),
                    ("Adjusted emissions growth rate for black carbon", 0.00553637799457393),
                    ("Growth rate in carbon dioxide emissions from land cover", 0.00830404102017226),
                    ("Greenhouse gas intensity growth rate", 0.084710896284828),
                    ("Greenhouse gas emissions per capita", 0.00553637799457393),
                ],
            },
        ],
    },
]

Ici on a généré un dictionnaire avec le tableur Excel et un premier code d'affichage de graphiques camemberts (qu'on a ensuite supprimé), qui représente la hiérarchie des c, des sc et des poids des indicateurs.

En bas s'affiche une représentation purement visuelle des changements qu'on a effectué sur les poids.

In [ ]:
# Flatten data across all 3 levels using Final weights
total_weight = sum(
    val for obj in hierarchy for grp in obj["groups"] for _, val in grp["vars"]
)

obj_angles, obj_starts, obj_colors, obj_labels = [], [], [], []
grp_angles, grp_starts, grp_colors, grp_labels = [], [], [], []
var_angles, var_starts, var_colors, var_labels = [], [], [], []

cursor = 0.0
for obj in hierarchy:
    obj_start = cursor
    obj_total = 0.0
    for grp in obj["groups"]:
        grp_start = cursor
        grp_total = 0.0
        for var_name, var_val in grp["vars"]:
            angle = (var_val / total_weight) * 2 * np.pi
            var_starts.append(cursor)
            var_angles.append(angle)
            var_colors.append(grp["var_color"])
            var_labels.append(f"{var_name} ({var_val * 100:.1f}%)")
            cursor += angle
            grp_total += var_val
        grp_starts.append(grp_start)
        grp_angles.append((grp_total / total_weight) * 2 * np.pi)
        grp_colors.append(grp["grp_color"])
        grp_labels.append(f"{grp['group']} ({grp_total * 100:.1f}%)")
        obj_total += grp_total
    obj_starts.append(obj_start)
    obj_angles.append((obj_total / total_weight) * 2 * np.pi)
    obj_colors.append(obj["obj_color"])
    obj_labels.append(f"{obj['objective']} ({obj_total * 100:.1f}%)")

fig, ax = plt.subplots(figsize=(11, 8), subplot_kw=dict(projection="polar"))

# Ring radii (bottom and radial thickness)
inner_bottom, inner_height = 0.25, 0.25
mid_bottom, mid_height = 0.50, 0.25
outer_bottom, outer_height = 0.75, 0.25

# 1. Inner ring: Policy Objectives
ax.bar(
    x=obj_starts,
    width=obj_angles,
    bottom=inner_bottom,
    height=inner_height,
    color=obj_colors,
    edgecolor="w",
    linewidth=1.2,
    align="edge",
)

# 2. Middle ring: Groups
ax.bar(
    x=grp_starts,
    width=grp_angles,
    bottom=mid_bottom,
    height=mid_height,
    color=grp_colors,
    edgecolor="w",
    linewidth=1.0,
    align="edge",
)

# 3. Outer ring: Variables
ax.bar(
    x=var_starts,
    width=var_angles,
    bottom=outer_bottom,
    height=outer_height,
    color=var_colors,
    edgecolor="w",
    linewidth=0.8,
    align="edge",
)

# Add labels inside larger slices
for start, width, label in zip(obj_starts, obj_angles, obj_labels):
    if width > 0.35:
        mid_angle = start + width / 2
        short_name = label.split(" (")[0].split()[0]
        ax.text(
            mid_angle,
            inner_bottom + inner_height / 2,
            short_name,
            ha="center",
            va="center",
            fontsize=8,
            fontweight="bold",
            color="#111111",
        )

for start, width, label in zip(grp_starts, grp_angles, grp_labels):
    if width > 0.45:
        mid_angle = start + width / 2
        pct = label.split("(")[-1].rstrip(")")
        ax.text(
            mid_angle,
            mid_bottom + mid_height / 2,
            pct,
            ha="center",
            va="center",
            fontsize=7.5,
            color="#111111",
        )

ax.set_ylim(0, 1.05)
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
ax.set_axis_off()
ax.set_title("EPI Performance Indicator Weights (Nested Hierarchy)", pad=20, fontsize=13)

plt.tight_layout()
plt.show()

### C. Cartographie de l’EPI officiel

Avant d’approfondir notre analyse, il est intéressant d’avoir une première vision globale de la répartition géographique des scores de l’EPI en 2022. Importons les bibliothèques nécessaires et les fonctions de cartographie implémentées dans le fichier `niels/map.py`.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from niels.map import (
    carte_statique_pays,
    carte_interactive_pays,
    carte_interactive_regions
)

#### Chargement des données

Commençons par charger les données nécessaires à la construction de nos cartes. En complément des données de l'EPI 2022, nous utilisons la base de données géographiques [Natural Earth](https://www.naturalearthdata.com/) à l'échelle 1:50 millions (1 cm représente 500 km dans la réalité).

Parmi les variables de cette base, trois nous intéressent particulièrement : les frontières des pays (`geometry`), leurs noms (`ADMIN`) et leurs codes à trois lettres (`ADM0_A3`).

In [ ]:
df_original = pd.read_excel(
    "raw_data/2022-epi.xlsx",
    sheet_name="3_EPI_Results"
)
df = df_original[["iso", "country", "region", "EPI.new"]]

url = (
    "https://naturalearth.s3.amazonaws.com/"
    "50m_cultural/ne_50m_admin_0_countries.zip"
)
world = gpd.read_file(url)

# Aperçu des principales variables de la base Natural Earth
world[["ADMIN", "ADM0_A3", "geometry"]].head()

#### Jointure des données

Maintenant que nos deux bases sont chargées, associons à chaque pays son score EPI et ses frontières géographiques. Nous effectuons une jointure à gauche à partir des codes à trois lettres (`ADM0_A3` et `iso`), qui permettent d’identifier les pays dans les deux bases (par exemple, **FRA** pour la France et **DEU** pour l’Allemagne).

In [ ]:
carte = world.merge(
    df,
    left_on="ADM0_A3",
    right_on="iso",
    how="left"
)


print("Nombre de pays dans la base EPI :", df["iso"].nunique())
print("Nombre de pays retrouvés après la jointure :", carte["iso"].nunique())
print("Nombre de territoires sans score EPI :", carte["EPI.new"].isna().sum())

carte[["ADMIN", "ADM0_A3", "region", "EPI.new", "geometry"]].head()

Nous retrouvons bien les 180 pays étudiés par l’EPI. Les 62 valeurs manquantes correspondent à des territoires supplémentaires de Natural Earth, sans score EPI dans notre fichier. 

Nous pouvons désormais passer à la représentation cartographique de ces données.  




#### Carte statique par pays

Commençons par une carte statique à l’échelle mondiale. Les pays sont colorés en fonction de leur score EPI : plus le vert est foncé, meilleure est leur performance environnementale. Les territoires sans score individuel restent en gris.

In [ ]:
fig = carte_statique_pays(carte)
plt.show()

#### Carte interactive par pays

Nous proposons également une version interactive de cette carte, qui permet de zoomer et de consulter le score EPI de chaque pays en le survolant.

In [ ]:
m = carte_interactive_pays(carte)
m

#### Carte interactive par région

Il est intéressant de regarder les performances environnementales à l’échelle des régions, qui regroupent des pays partageant certaines caractéristiques géographiques, économiques ou climatiques. Nous calculons le score EPI moyen de chaque région en accordant le même poids à chaque pays.

In [ ]:
moyennes_regions = df.groupby("region")["EPI.new"].mean()

carte2 = carte.copy()
carte2["EPI_moyen_region"] = carte2["region"].map(moyennes_regions)
carte2.loc[carte2["EPI.new"].isna(), "EPI_moyen_region"] = float("nan")

moyennes_regions.sort_values(ascending=False).round(2).to_frame(
    name="Score EPI moyen"
)

In [ ]:
m2 = carte_interactive_regions(carte2)
m2

Ces cartes montrent de fortes disparités entre les régions du monde, avec des scores EPI généralement plus élevés en Europe occidentale et plus faibles en Afrique et en Asie du Sud.
Dans la partie suivante, nous chercherons à mieux comprendre ces différences et à questionner la manière dont l’EPI officiel est construit.

# **II. Analyse critique de l'EPI officiel**
## A. Lien entre richesse et performance environnementale (économétrie)
Partie econometrie : est-ce que l'EPI c'est juste la richesse du pays ?

In [ ]:
import numpy as np
import statsmodels.formula.api as smf
from martin.chargement import charger_donnees, CATEGORIES

df = charger_donnees()
df["log_pib"] = np.log(df["pib_hab"])  # en log sinon le Qatar, le Luxembourg etc écrasent tout

# --- 1) MCO simple EPI ~ log(pib/hab)
# HC1 = ecarts types robustes (l'equivalent du ", robust" sous Stata)
reg1 = smf.ols("EPI ~ log_pib", data=df).fit(cov_type="HC1")
print(reg1.summary())

residu = EPI observé - EPI prédit par la richesse
=> en gros ca mesure l'effort environnemental du pays a richesse donnée (c'est ça qui est intéressant)

In [ ]:
df["residu"] = reg1.resid
colonnes = ["country", "pib_hab", "EPI", "residu"]
print("moins bien que ce que leur richesse prédit :")
print(df.sort_values("residu")[colonnes].head(10))
print("mieux que prévu :")
print(df.sort_values("residu")[colonnes].tail(10))

2) dans l'autre sens : log(pib) sur les 11 categories
pas causal du tout, c'est juste pour voir quel profil ont les pays riches
rq : statsmodels vire tout seul les pays sans score de peche (pays sans littoral) -> N = 127
on avait essayé avec les 40 indicateurs mais trop de variables pour trop peu de pays, ça marchait pas

In [ ]:
reg2 = smf.ols("log_pib ~ " + " + ".join(CATEGORIES), data=df).fit(cov_type="HC1")
print(reg2.summary())

## B. Diversité des profils environnementaux (K-means)
clustering des pays avec k-means sur les 11 categories
idée : voir s'il y a des "profils" de pays et si ça colle avec les continents

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from martin.chargement import charger_donnees, CATEGORIES

df = charger_donnees()

kmeans plante s'il y a des NaN -> on remplace par la moyenne de la colonne
(c'est surtout FSH, la peche, qui manque pour les 44 pays sans acces a la mer. pas parfait mais bon)

In [ ]:
X = df[CATEGORIES].fillna(df[CATEGORIES].mean())
# pas de standardisation, tout est deja entre 0 et 100

# methode du coude pour choisir k
inerties = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inerties.append(km.inertia_)
plt.plot(range(1, 11), inerties, marker="o")
plt.xlabel("nombre de groupes k")
plt.ylabel("inertie (distance des pays au centre de leur groupe)")
plt.title("Méthode du coude")
plt.show()
# ça se casse vers k=4 donc on part là dessus

df["groupe"] = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(X)

# moyenne de chaque categorie par groupe, c'est ce qui permet de comprendre qui est qui
profils = df.groupby("groupe")[CATEGORIES + ["EPI", "pib_hab"]].mean().round(0)
print(profils.to_string())

# est ce qu'on retrouve les regions ?
print(pd.crosstab(df["region"], df["groupe"]))

for g in range(4):
    print("Groupe", g, ":", ", ".join(df[df["groupe"] == g]["country"]))

# heatmap des profils (lignes = groupes, colonnes = categories)
plt.imshow(profils[CATEGORIES], cmap="Greens")
plt.xticks(range(len(CATEGORIES)), CATEGORIES)
plt.yticks(range(4), ["groupe " + str(g) for g in range(4)])
plt.colorbar(label="score moyen (0-100)", shrink=0.5)
plt.title("Profil moyen de chaque groupe")
plt.show()

carte du monde
contours des pays = natural earth, lu direct en ligne
le /vsizip/vsicurl/ devant c'est parce que sur mon mac python refusait l'url (erreur de certificat SSL)

In [ ]:
monde = gpd.read_file("/vsizip/vsicurl/https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")
monde = monde[monde["ADM0_A3"] != "ATA"]  # pas besoin de l'antarctique
# merge sur le code pays. attention prendre ADM0_A3 et pas ISO_A3 (France et Norvege y sont a -99 !)
monde = monde.merge("groupe " + df["groupe"].astype(str), left_on="ADM0_A3", right_index=True, how="left")
ax = monde.plot(color="white", edgecolor="lightgrey", figsize=(12, 6))  # en blanc = pas noté dans l'EPI
monde.dropna(subset=["groupe"]).plot(column="groupe", categorical=True, cmap="tab10", legend=True, ax=ax)
ax.set_title("Groupes K-means des pays (11 catégories de l'EPI)")
ax.set_axis_off()
plt.show()

## C. Structure statistique et redondance des catégories (ACP)
ACP sur les 11 categories
les poids de l'EPI sont choisis par des experts -> et si on laissait les données choisir ?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from chargement import charger_donnees, CATEGORIES

df = charger_donnees()
X = df[CATEGORIES].fillna(df[CATEGORIES].mean())  # meme astuce que pour le kmeans (NaN de la peche)

# ACP normée (on centre-réduit, comme en cours) sinon les categories qui varient beaucoup prennent tout
Z = StandardScaler().fit_transform(X)
acp = PCA().fit(Z)

# eboulis des valeurs propres : combien d'axes il faut pour résumer les pays ?
plt.bar(range(1, 12), acp.explained_variance_ratio_ * 100)
plt.xlabel("axe")
plt.ylabel("% de variance expliquée")
plt.title("Eboulis des valeurs propres")
plt.show()
print("variance expliquée par les axes 1 et 2 :", acp.explained_variance_ratio_[:2].round(2))

# poids officiels d'une categorie = son poids x le poids de son objectif (feuille 6)
w = pd.read_excel("raw_data/2022-epi.xlsx", sheet_name="6_EPI_Weights").set_index("Abbreviation")
poids = pd.DataFrame(index=CATEGORIES)
poids["officiel"] = [w.loc[c, "Weight"] * w.loc[w.loc[c, "NextLevel"], "Weight"] for c in CATEGORIES]

# poids "ACP" = coefficients de l'axe 1 ramenés à une somme de 1
# (le signe d'un axe est arbitraire, on le retourne si besoin pour que ce soit positif)
axe1 = acp.components_[0] * np.sign(acp.components_[0].sum())
poids["acp"] = axe1 / axe1.sum()
print(poids.round(3))

# nouvel indice avec les poids de l'ACP, puis on compare les classements
df["epi_acp"] = X @ poids["acp"]
df["rang_officiel"] = df["EPI"].rank(ascending=False)
df["rang_acp"] = df["epi_acp"].rank(ascending=False)
df["places_gagnees"] = df["rang_officiel"] - df["rang_acp"]
print("corrélation des rangs :", round(df["rang_officiel"].corr(df["rang_acp"]), 2))
colonnes = ["country", "EPI", "rang_officiel", "rang_acp", "places_gagnees"]
print(df.sort_values("places_gagnees")[colonnes].head(10))  # ceux qui perdent le plus
print(df.sort_values("places_gagnees")[colonnes].tail(10))  # ceux qui gagnent le plus

# plan factoriel (axes 1 et 2), en couleur les groupes du kmeans (memes reglages que clustering.py)
groupes = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(X)
coord = acp.transform(Z)
plt.scatter(coord[:, 0], coord[:, 1], c=groupes, cmap="tab10")
for pays in ["FRA", "USA", "CHN", "IND", "QAT", "SAU", "TUR", "DNK", "BRA", "NER"]:
    i = df.index.get_loc(pays)
    plt.annotate(pays, (coord[i, 0], coord[i, 1]))
plt.xlabel("axe 1 (" + str(round(acp.explained_variance_ratio_[0] * 100)) + " %)")
plt.ylabel("axe 2 (" + str(round(acp.explained_variance_ratio_[1] * 100)) + " %)")
plt.title("Plan factoriel, couleur = groupe kmeans")
plt.show()

ce qu'on en retient :
- attention l'ACP donne pas un "meilleur" EPI, elle surpondère les categories redondantes (double comptage)
- les 11 categories c'est pas 11 infos independantes : un gros bloc "developpement" (axe 1, 46%)
  + une dimension plutot geographique (axe 2, les pays arides)
- les experts mettent ~23% sur ce bloc et 38% sur le climat -> ca neutralise le double comptage
  et donc le poids de la richesse (le papier le justifie pas comme ça mais c'est ce que ça fait)
- selon la façon de pondérer certains pays bougent de 90 places, le classement dépend beaucoup
  d'un choix que le papier présente comme technique


# III. Analyse spatiale et temporelle des performances environnementales

Pour étudier l'évolution des performances environnementales entre 1995 et 2022, nous allons reconstituer une base de données historique. La reconstruction présente trois limites principales :

- **Les valeurs manquantes :** certains indicateurs n'ont pas été mesurés pour certains pays ou certaines années. 

- **Les indicateurs non applicables :** certaines valeurs sont absentes car l'indicateur ne s'applique pas au pays concerné, comme les indicateurs liés à la pêche pour les pays sans accès à la mer.

- **L'hétérogénéité temporelle :** les méthodes de mesure des indicateurs ont pu évoluer au cours des dernières décennies, limitant  la comparabilité des valeurs entre les années.

Malgré ces limites, nous adoptons une méthodologie cohérente sur l'ensemble de la période étudiée afin de comparer:

- **L'EPI historique**, calculé à partir des pondérations officielles de l'EPI 2022.

- **Le nouvel EPI historique**, calculé à partir des indicateurs et pondérations retenus dans la partie précédente.

L'EPI global constituant le principal outil de synthèse susceptible d'orienter les décisions des pouvoirs publics, nous choisissons de ne pas approfondir l'évolution des sous-indices intermédiaires.

## A. Construction et nettoyage des bases historiques (1995-2022)

### Construction de la base historique

Nous fusionnons les 40 fichiers CSV contenant les données historiques des indicateurs (disponibles dans `raw_data/2022-epi-indicators-time-series-na`) à l'aide de la fonction `construire_base_historique` (implémentée dans `niels/dynamique.py`) qui effectue des jointures successives.

In [ ]:
import pandas as pd

# Importation des fonctions implémentées dans le fichier niels/dynamique.py
from niels.dynamique import (
    construire_base_historique,
    ajouter_regions,
    imputer_valeurs_manquantes,
    charger_ponderations_nouvelles,
    charger_ponderations_officielles,
    calculer_epi
)

dossier = "raw_data/2022-epi-indicators-time-series-na"
chemin_excel = "raw_data/2022-epi.xlsx"
chemin_poids = "niels/clear_indices.xlsx"

df_global = construire_base_historique(dossier)
df_global.head()

Nous conservons les 180 pays présents dans l'EPI 2022 et associons à chacun sa région géographique à l'aide de la fonction `ajouter_regions`. 

In [ ]:
df_etude_na = ajouter_regions(df_global, chemin_excel)
print("Nombre de pays :", df_etude_na["iso"].nunique())
print("Nombre de lignes :", len(df_etude_na))
df_etude_na.head()

### Traitement des valeurs manquantes

Les auteurs du rapport ont déjà traité une partie des données manquantes par interpolation linéaire et prolongement constant aux extrémités des séries. Pour les indicateurs non applicables, ils redistribuent leurs poids entre les autres indicateurs.
Par souci de simplicité, nous choisissons de remplacer les valeurs manquantes restantes par la moyenne régionale de l'indicateur pour l'année considérée (`imputer_valeurs_manquantes`). Les pondérations restent ainsi constantes, mais le classement de 2022 pourra différer du classement officiel.

In [ ]:
df_etude = imputer_valeurs_manquantes(df_etude_na)
indicateurs = df_etude.columns.drop(["code", "iso", "country", "year", "region"])
print("Valeurs manquantes avant imputation :", df_etude_na[indicateurs].isna().sum().sum())
print("Valeurs manquantes après imputation :", df_etude[indicateurs].isna().sum().sum())

### Calcul des indices historiques

Notre base historique est désormais prête. Nous récupérons les pondérations officielles et les nouvelles pondérations, puis calculons les deux indices pour chaque pays et chaque année à l'aide de la fonction `calculer_epi`.

In [ ]:
poids_officiels = charger_ponderations_officielles(chemin_excel)
poids_nouveaux = charger_ponderations_nouvelles(chemin_poids, chemin_excel)

print("Nombre d'indicateurs officiels :", len(poids_officiels))
print("Nombre d'indicateurs retenus :", len(poids_nouveaux))

df_final = calculer_epi(df_etude, poids_officiels, "EPI_officiel")
df_final = calculer_epi(df_final, poids_nouveaux, "EPI_new")

df_final[["country", "region", "year", "EPI_officiel", "EPI_new"]].head()


## B. Analyse et comparaison des trajectoires environnementales selon les deux indices

À partir de la base historique précédemment construite, nous étudions l’évolution des performances environnementales entre 1995 et 2022. Les fonctions de visualisation sont implémentées dans le fichier `niels/dynamique_graphique.py`.

In [ ]:
from niels.dynamique_graphique import (
    graphique_evolution_epi_moyen_median_mondial,
    graphique_evolution_epi_moyen_par_region,
    graphique_classement_variation_epi_par_pays,
    graphique_evolution_classement_par_region,
    tableau_synthese_evolution_par_region
)

Avant de comparer les deux indices, nous vérifions la disponibilité des scores pour chaque année.

In [ ]:
df_final.groupby("year")[["EPI_officiel", "EPI_new"]].count()

### Évolution des performances mondiales

Nous observons tout d’abord l’évolution des scores moyens et médians à l’échelle mondiale.

In [ ]:
graphique_evolution_epi_moyen_median_mondial(
    df_final, "EPI officiel", "EPI_officiel"
)

graphique_evolution_epi_moyen_median_mondial(
    df_final, "Nouvel EPI", "EPI_new"
)

Les deux indices mettent en évidence une amélioration globale des performances environnementales entre 1995 et 2022. Cette progression est plus marquée pour l'EPI officiel, dont le score moyen augmente d'environ 6,5 points, contre 5 points pour notre nouvel indice. On remarque que l'écart entre la moyenne et la médiane est plus important pour le nouvel EPI, ce qui suggère une distribution des scores plus asymétrique.

### Évolution des performances régionales

Nous étudions ensuite les trajectoires moyennes des régions.

In [ ]:
graphique_evolution_epi_moyen_par_region(
    df_final, "EPI officiel", "EPI_officiel"
)

graphique_evolution_epi_moyen_par_region(
    df_final, "Nouvel EPI", "EPI_new"
)

Les nouvelles pondérations accentuent les disparités régionales : en 2022, près de 39 points séparent les pays occidentaux de l'Afrique subsaharienne, contre 27 points avec l'EPI officiel. Cela pourrait s'expliquer par le fait que les pays développés disposent de davantage de moyens pour répondre aux besoins environnementaux humains. On remarque que les anciens États soviétiques régressent avec l'EPI officiel, mais progressent avec notre indice. On peut supposer que l'amélioration des conditions de vie dans ces pays ne s'est pas nécessairement accompagnée d'une amélioration de l'ensemble des dimensions environnementales. Ces hypothèses mériteraient toutefois d'être vérifiées à l'échelle des sous-indices intermédiaires.

### Évolution des performances nationales

Nous identifions les dix pays ayant le plus progressé et les dix ayant le moins progressé entre 1995 et 2022, selon chaque indice.

In [ ]:
graphique_classement_variation_epi_par_pays(
    df_final, "EPI officiel", "EPI_officiel"
)

graphique_classement_variation_epi_par_pays(
    df_final, "Nouvel EPI", "EPI_new"
)

Si certains pays, comme Malte, le Danemark et la Suède, figurent parmi les meilleures progressions selon les deux indices, les classements diffèrent sensiblement. Avec les nouvelles pondérations, des pays comme le Turkménistan, l'Ouzbékistan et l'Ukraine (anciens États soviétiques) apparaissent parmi les plus fortes progressions, ce qui rejoint nos observations à l'échelle régionale. À l'inverse, les pays d'Afrique subsaharienne sont davantage représentés parmi les plus fortes régressions. Une explication possible serait que les progrès en matière de santé environnementale et d'accès aux ressources essentielles ont été inégalement répartis entre les pays. Notre indice permet ainsi de mettre en lumière des évolutions que l'EPI officiel tend à atténuer.

### Bilan

L'analyse dynamique montre que, si l'amélioration globale des performances environnementales reste observable avec les deux indices, les conclusions à l'échelle régionale et nationale dépendent davantage des pondérations retenues. Pour compléter cette analyse, nous comparons l'évolution du classement international moyen de chaque région entre 1995 et 2022.

In [ ]:
tableau_synthese_evolution_par_region(df_final)

Le cas des anciens États soviétiques est particulièrement révélateur : ils perdent en moyenne 51 places avec l'EPI officiel, mais en gagnent 3 avec notre indice. À l'inverse, l'Europe de l'Est voit sa progression se transformer en recul. Les choix de construction d'un indice influencent non seulement notre perception des performances environnementales, mais aussi celle des progrès réalisés au cours du temps. Ces résultats restent  à nuancer compte tenu des limites de la reconstruction historique et du traitement des valeurs manquantes.